# Until Day 7 - everything I've learned, in one place

This is my study file for Days 1-7. Up to now, Claude helped me write the answers. From Day 8 on I write them myself, so this file is for **actually learning** the material, not just having it.

## How to use this file

1. **Read** one section.
2. **Close it** and explain the idea out loud (or on paper) in your own words, as if teaching a friend.
3. Try every **Q:** before clicking **Show answer**. Getting it wrong first and then checking is how memory sticks.
4. For coding problems, **re-write the solution from a blank file** without looking. If you get stuck, peek, close, and try again tomorrow.
5. Come back to the **Final self-test** at the end every few days.

**Rule from tomorrow:** attempt first, check second. A wrong answer I wrote myself teaches more than a right answer I copied.

For runnable demos of every idea, see the per-day notebooks `01_...` to `07_...` in this folder.

## Contents

- Day 1 - Project setup: uv, lock files, secrets
- Day 2 - Calling LLMs and measuring them honestly
- Day 3 - Streaming answers with SSE
- Day 4 - Hash sets, hash maps, and how LLMs are made
- Day 5 - Tokens, sampling, and cost
- Day 6 - Middleware, logging to Postgres, testing with fakes
- Day 7 - Anagrams: counting and canonical keys
- Patterns that keep coming back
- Final self-test

---
# Day 1 - Project setup: uv, lock files, secrets

**Big idea:** make the project *reproducible* (it works the same on every machine) and *safe* (secrets never leak).

## What I set up
- A git repo on GitHub, managed with **uv**.
- `pyproject.toml` + `uv.lock`, a `.env` for secrets (git-ignored) and `.env.example` (committed).
- My goals for the next months.

## The concepts

**uv** is a fast tool that installs Python packages and manages the project's virtual environment (`.venv`).

| Command | What it does |
|---|---|
| `uv add fastapi` | adds a dependency and updates `pyproject.toml` and `uv.lock` |
| `uv add --dev pytest` | adds a dependency used only for development/testing |
| `uv sync` | installs exactly what `uv.lock` says |
| `uv sync --frozen` | same, but never re-resolves or rewrites the lock file (used in Docker) |
| `uv run <cmd>` | runs a command inside the project's environment |

**`pyproject.toml` vs `uv.lock` - shopping list vs receipt.**
- `pyproject.toml` says what you *want*: "fastapi, version 0.141.1 or newer."
- `uv.lock` records exactly what you *got*: "fastapi 0.141.1, starlette 1.6.0, anyio 4.15.1, ..." - including **transitive dependencies** (packages your packages need, which you never asked for).
- Without the lock, "or newer" can mean a different version next month, and a surprise update can break the app. With it, everyone gets the identical environment. **Commit both.**

**Secrets.**
- Keys go in `.env`. Code reads them with `load_dotenv()` + `os.getenv("KEY")`.
- `.env` is listed in `.gitignore`, so git never tracks it.
- `.env.example` has the same variable names with empty values, and *is* committed, so others know what to fill in.
- Never print a key, not even part of it.

**Git is a photo album, not a whiteboard.** Every commit is a permanent snapshot. Deleting a file in a later commit doesn't remove it from the earlier snapshots.

**Q: In your own words: what does uv.lock protect you from that pyproject.toml alone doesn't?**

<details>
<summary>Show answer</summary>

`pyproject.toml` only gives version *ranges*, so the resolver can pick different versions on different days or machines, including for transitive dependencies. `uv.lock` pins the exact version of every package in the whole tree, so `uv sync` rebuilds the same environment every time. It protects you from "it worked yesterday / on my machine" breakage caused by a package silently updating.

</details>

**Q: You committed .env and pushed it. You delete it in the next commit. Are the secrets safe?**

<details>
<summary>Show answer</summary>

No. The old commit still contains the file, and anyone who cloned, forked, or scraped the repo may already have it. The fix, **in this order**:
1. **Rotate/revoke every key** in that file. This is the only step that actually protects you.
2. Remove the file from history (`git filter-repo` or BFG) and force-push.
3. Make sure `.env` is in `.gitignore`.
4. Treat the old keys as compromised forever, even after cleaning history.

</details>

## My goals (by 24 Jan 2027)
1. Ship a production-ready AI backend (FastAPI, PostgreSQL, Redis, auth, background jobs, Docker), deployed publicly.
2. Build a multi-tenant RAG API with document ingestion, retrieval, citations, tenant isolation, and an automated eval suite in CI.
3. A portfolio of 3 deployed AI projects, each with tests, docs, Docker, eval metrics, and a demo I can explain in an interview.

## Test yourself

**Q: Which file do you commit: .env or .env.example? Why?**

<details>
<summary>Show answer</summary>

`.env.example` (names only, no real values) so others know which variables to set. `.env` holds real secrets and stays out of git via `.gitignore`.

</details>

**Q: What's a transitive dependency? Give an example from this project.**

<details>
<summary>Show answer</summary>

A package you didn't ask for, pulled in because something you asked for needs it. Example: I added `fastapi`, which needs `starlette` and `anyio`.

</details>

**Q: Why does the Dockerfile use `uv sync --frozen`?**

<details>
<summary>Show answer</summary>

So the image installs exactly what's in `uv.lock` and never re-resolves or rewrites the lock during the build. The container gets the same versions I tested with.

</details>

---
# Day 2 - Calling LLMs and measuring them honestly

**Big idea:** calling a model is easy. Knowing how fast and how expensive it *really* is takes care.

## What I built
- `test_connections.py` - checks my Gemini key and local Ollama both answer "pong".
- `compare_llms.py` - sends the same prompt to Gemini, OpenRouter, and Ollama, 3 times each, and prints median latency, tokens, and cost.

| Provider | Model | Median latency | In tok | Out tok | Cost |
|---|---|---|---|---|---|
| Gemini | gemini-3.6-flash | 4.39 s | 12 | 37 | $0.000148 |
| OpenRouter | deepseek-v4-flash (free) | 2.39 s | 94 | 50 | $0 |
| Ollama | llama3.2:3b (local) | 1.69 s | 36 | 43 | $0 |

## The concepts

**An LLM API call** = send a prompt, get back text + **usage** (input and output token counts). Every provider has this shape; only the URL, auth header, and field names differ.

**Measuring time:** `time.perf_counter()` is a stopwatch. Read it before, read it after, subtract.

**Median of 3, not one run.** One run can be unlucky (cold connection, busy server). The **average** gets dragged by one bad run; the **median** (middle value after sorting) ignores it. `[1.21, 1.34, 9.80]` -> average 4.12 s, median 1.34 s.

**Cost formula** (prices are per 1 million tokens, input and output priced separately):

```
cost = input_tokens / 1,000,000 x input_price  +  output_tokens / 1,000,000 x output_price
```

- Gemini: I multiplied tokens by the price from Google's pricing page.
- OpenRouter: the response itself reports the real cost (`usage.cost`).
- Ollama: runs on my laptop, so $0.

**`max_tokens` is a ceiling, not a purchase.** You pay for tokens actually generated. `max_tokens=4000` for a 20-token answer costs the same as `max_tokens=20`; the limit just stops a runaway answer sooner.

**End-to-end latency is not model speed.** Ollama "won" mostly because it skips the internet and is a tiny 3B model, not because its model computes faster. A fair comparison would remove network time and match hardware and model size.

### Key code (from `compare_llms.py`, simplified)
```python
def compute_cost(input_tokens, output_tokens, price_in_per_1m, price_out_per_1m):
    return (input_tokens / 1_000_000) * price_in_per_1m + (output_tokens / 1_000_000) * price_out_per_1m

def run_with_median(call_fn, prompt, runs=3):
    attempts = [call_fn(prompt) for _ in range(runs)]
    ok = [r for r in attempts if r.error is None]
    return statistics.median(r.latency_s for r in ok)   # plus median tokens and cost
```

## Questions I was asked

**Q: Why can two providers report similar token counts but charge different amounts?**

<details>
<summary>Show answer</summary>

Token count and price are separate things. Cost = tokens x **rate**, and rates differ between providers and models (and between input and output, cached vs uncached input, context tiers). 1,000 tokens at $1/M vs $5/M is a 5x different bill for the same usage.

</details>

**Q: What happens if you set max_tokens=4000 for a one-line answer?**

<details>
<summary>Show answer</summary>

Nothing extra on the bill: you pay only for tokens actually generated. It's a maximum, not a request to generate 4,000. The downside is a looser safety rail: if the model rambles, it's allowed to run (and cost, and take) much longer before being cut off. For a one-line answer something like 100 is more sensible.

</details>

**Q: Ollama had the lowest latency. Why, and does it prove its model is faster?**

<details>
<summary>Show answer</summary>

Mostly because it runs locally: no network round trip to a remote provider. It doesn't prove the model computes faster: I measured end-to-end time, and it's a small 3B model on my hardware vs larger models on someone else's GPUs. To compare raw speed you'd isolate inference time and match hardware and model size.

</details>

## Test yourself

**Q: A model charges $3/M input and $15/M output. A call uses 2,000 input and 500 output tokens. What does it cost?**

<details>
<summary>Show answer</summary>

Input: 2,000 / 1,000,000 x $3 = $0.006.
Output: 500 / 1,000,000 x $15 = $0.0075.
**Total: $0.0135.**

</details>

**Q: Latencies were 0.8 s, 0.9 s, 6.0 s. Average vs median - which do you report, and why?**

<details>
<summary>Show answer</summary>

Average = 2.57 s, median = 0.9 s. Report the **median**: the 6.0 s run is an outlier (cold start, network blip) and the median shows what a typical request feels like.

</details>

In [1]:
# Try it: change the numbers and predict the answer before running.
def cost(input_tokens, output_tokens, price_in_per_1m, price_out_per_1m):
    return input_tokens / 1_000_000 * price_in_per_1m + output_tokens / 1_000_000 * price_out_per_1m

print(f"${cost(2_000, 500, 3.00, 15.00):.4f}")

$0.0135


---
# Day 3 - Streaming answers with SSE

**Big idea:** users don't feel *total* time, they feel *how long they stared at a blank screen*. Streaming shows the first words right away.

## What I built
- `days/day-003/app.py` - FastAPI `GET /chat?prompt=...&provider=ollama|gemini` that streams the answer with `EventSourceResponse`.
- `days/day-003/test_chat.py` - one pytest test using `httpx.AsyncClient` + `ASGITransport` (calls the app in-process, no real network socket).
- `days/day-003/Dockerfile` - `python:3.12-slim`, `uv sync --frozen --no-dev`, `EXPOSE 8000`, `CMD ["uvicorn", "app:app", ...]`. Built from the repo root so `pyproject.toml` and `uv.lock` are in the build context.

Run it: `uv run uvicorn app:app --app-dir days/day-003 --reload`

## The concepts

**TTFT (time to first token)** is what users feel. Same 5-second answer: first word at 0.2 s feels fast; nothing for 4.5 s then everything at once feels broken.

**async / await.** `await x` = "this takes a while, go do other work and come back." One event loop juggles many requests while they wait on the network. An `async def` with `yield` is an **async generator**: it hands out values one at a time. Perfect for tokens.

**SSE (Server-Sent Events)** = plain text over a normal HTTP response with `Content-Type: text/event-stream`. Each message is `field: value` lines ending with a blank line:
```
event: message
data: Hello

```

**Streaming from Ollama:** `httpx.AsyncClient().stream("POST", ...)` + `response.aiter_lines()`. Ollama sends one JSON object per line: `{"response": "Hel", "done": false}` ... `{"done": true}`.

**Timeouts.** `httpx.Timeout(connect=5, read=30, write=5, pool=5)`. On `httpx.TimeoutException`, yield **one final error chunk** and return cleanly instead of hanging.

**Disconnects - the big gotcha.** My `request.is_disconnected()` check never fired. Why: `EventSourceResponse` runs its own listener that reads the one-time ASGI "disconnect" message first, and a message can only be read once. What sse-starlette does instead is **cancel the generator's task**, which raises `asyncio.CancelledError` inside it. So: catch `CancelledError`, clean up, and **re-raise** it.

**Swapping providers without touching the streaming code.** Each provider is just "an async generator of text". Put them in a dict (`TOKEN_SOURCES`) and look one up by name. One shared wrapper handles SSE framing, timeouts, and disconnects for all of them. Unknown name -> `400`.

### Key code (the shared wrapper)
```python
TOKEN_SOURCES = {"ollama": ollama_tokens, "gemini": gemini_tokens}

async def sse_stream(token_source):
    try:
        async for token in token_source:
            yield {"event": "message", "data": token}
    except httpx.TimeoutException:
        yield {"event": "error", "data": "upstream request timed out"}
        return
    except asyncio.CancelledError:
        print("Client disconnected, stopping stream")
        raise                                   # always re-raise a cancel

@app.get("/chat", response_class=EventSourceResponse)
async def chat(prompt: str, provider: str = "ollama"):
    source = TOKEN_SOURCES.get(provider)
    if source is None:
        raise HTTPException(status_code=400, detail="unknown provider")
    return EventSourceResponse(sse_stream(source(prompt)))
```

## Questions I was asked

**Q: Why does time-to-first-token matter more than total latency for a chat UI, even when total latency is unchanged?**

<details>
<summary>Show answer</summary>

Total latency is what the backend measures; TTFT is what the person experiences. Once words start appearing, the app feels alive and the user starts reading while the rest streams in. With identical total time, a response that starts at 200 ms feels fast; one that shows nothing for 4.5 s and then dumps everything feels frozen, and the user wonders if it's broken.

</details>

**Q: What breaks if you never handle disconnects and a user closes the tab 3 seconds into a long generation? Who pays?**

<details>
<summary>Show answer</summary>

Nothing notices, so nothing stops: the generator keeps pulling tokens, the model keeps generating, and the server keeps writing to a socket nobody reads. With **Ollama** you pay in wasted GPU/CPU that other requests could have used. With a **paid API** you pay real money for every token generated after the user left. At scale, abandoned chats also hold connections and tasks open, wasting capacity.

</details>

**Q: What breaks if you drop the httpx timeout and Ollama silently stops responding mid-request?**

<details>
<summary>Show answer</summary>

That one request's task waits **forever** on `aiter_lines()`. Because it's async, other requests still get served, so nothing crashes. But the stuck task keeps its connection to Ollama, the browser's open SSE response, and its memory alive indefinitely. If it keeps happening, these pile up (a slow leak) until you run out of connections or file descriptors and the server can't accept new work. A self-inflicted outage caused only by a missing timeout.

</details>

### Recall quiz from Day 3 (about Day 2)

**Q: Why is the median of 3 runs better than a single run?**

<details>
<summary>Show answer</summary>

One run can be unlucky in either direction. The median isn't swayed by one outlier, so it reflects a typical run.

</details>

**Q: $1/M input, $5/M output, 500 input + 200 output tokens. Cost?**

<details>
<summary>Show answer</summary>

0.0005 + 0.001 = **$0.0015**.

</details>

**Q: Does Ollama's near-zero network overhead prove its model is computationally faster?**

<details>
<summary>Show answer</summary>

No. It measured end-to-end time; Ollama skips the network, and it's a different, smaller model on different hardware. Not a controlled comparison.

</details>

**Q: Are you billed for the unused part of max_tokens?**

<details>
<summary>Show answer</summary>

No. You pay for tokens actually produced.

</details>

## Test yourself

**Q: Write the exact bytes of one SSE message carrying the text 'hi'.**

<details>
<summary>Show answer</summary>

`event: message\r\ndata: hi\r\n\r\n` (sse-starlette uses `\r\n` line endings; a blank line ends the message. `event:` is optional.)

</details>

**Q: Why must you re-raise CancelledError after catching it?**

<details>
<summary>Show answer</summary>

Cancellation is asyncio asking the task to stop. If you swallow it, the task looks like it finished normally and asyncio's cancel doesn't complete properly. Catch it only to clean up or log, then `raise`.

</details>

**Q: Why does the Dockerfile set ENV PATH="/app/.venv/bin:$PATH"?**

<details>
<summary>Show answer</summary>

`uv sync` installs packages into `/app/.venv`. Putting its `bin` on PATH lets `CMD ["uvicorn", ...]` find uvicorn directly without `uv run`.

</details>

---
# Day 4 - Hash sets, hash maps, and how LLMs are made

**Big idea (coding):** when a problem asks "have I seen this before?", a set or dict turns a slow nested loop into one fast pass.

## Contains Duplicate (LeetCode 217)
*Does any value appear at least twice?*

- **Brute force:** compare every pair -> O(n^2) time.
- **Hash set:** walk once; if the number is already in `seen`, return True, else add it. Set lookups are O(1) on average -> **O(n) time, O(n) space.**
- **Sort first:** duplicates end up next to each other -> O(n log n) time, little extra space (but it changes the list).

```python
def contains_duplicate(nums):
    seen = set()
    for n in nums:
        if n in seen:
            return True
        seen.add(n)
    return False
```

## Two Sum (LeetCode 1)
*Return the indices of the two numbers that add up to `target`.*

For each number `n`, the partner you need is `target - n` (the **complement**). Keep a dict `value -> index` of everything seen so far. **Check before storing**, so a number can't pair with itself. **O(n) time, O(n) space.**

```python
def two_sum(nums, target):
    seen = {}                       # value -> index
    for i, n in enumerate(nums):
        complement = target - n
        if complement in seen:
            return [seen[complement], i]
        seen[n] = i
    return []
```

Walkthrough of `[2, 7, 11, 15]`, target 9: i=0, n=2, need 7, seen={} -> store {2:0}. i=1, n=7, need 2, 2 is in seen -> return [0, 1].

**Pattern:** need to know *whether* you've seen it -> **set**. Need to know *where* -> **dict**.

In [2]:
# Try it: predict each output first.
def contains_duplicate(nums):
    seen = set()
    for n in nums:
        if n in seen:
            return True
        seen.add(n)
    return False

def two_sum(nums, target):
    seen = {}
    for i, n in enumerate(nums):
        if target - n in seen:
            return [seen[target - n], i]
        seen[n] = i
    return []

print(contains_duplicate([1, 2, 3, 1]), contains_duplicate([1, 2, 3, 4]), contains_duplicate([]))
print(two_sum([2, 7, 11, 15], 9), two_sum([3, 2, 4], 6), two_sum([3, 3], 6))

True False False
[0, 1] [1, 2] [0, 1]


## Karpathy - "Intro to Large Language Models" (10 notes)

1. **An LLM is two files:** a huge parameters file (the weights; ~140 GB for Llama 2 70B) and a small program (~500 lines of C) that runs them.
2. **Training is expensive, inference is cheap:** thousands of GPUs for weeks and millions of dollars to train; much less to run.
3. **Training is lossy compression of the internet:** ~10 TB of text squeezed into the weights. Patterns survive, exact facts get blurry.
4. **So it hallucinates:** it predicts *plausible* next tokens; it doesn't look facts up.
5. **Two stages:** pretraining on internet text -> a "base model" that just continues documents; fine-tuning on curated Q&A conversations (optionally + RLHF) -> an assistant.
6. **Scaling laws:** more parameters + more data -> predictably better. That's what drives the race for compute.
7. **Tool use:** models call search, calculators, code interpreters instead of doing everything from memory.
8. **System 1 vs System 2:** today's models answer on reflex (System 1); the goal is slower, deliberate reasoning that spends more compute for better answers (System 2).
9. **LLM as an operating system kernel:** it coordinates resources - context window as memory, tools, retrieval.
10. **Security:** jailbreaks (talking past safety training), prompt injection (hidden instructions in content the model reads), data poisoning (planted triggers in training data). Still an open cat-and-mouse game.

## Test yourself

**Q: In Two Sum, why do we check for the complement BEFORE storing the current number?**

<details>
<summary>Show answer</summary>

So a number can't be paired with itself. For `[3, 3]` target 6 it still works: at i=1 we find the first 3 (index 0) already stored -> [0, 1]. But for `[3]` target 6 we must NOT return [0, 0].

</details>

**Q: Contains Duplicate: what do you trade by using a hash set instead of sorting?**

<details>
<summary>Show answer</summary>

**Memory for speed.** Set: O(n) time but O(n) extra memory. Sort: O(n log n) time with little extra memory, but it mutates the input (copying it costs O(n) memory again) and can't stop early at the first duplicate the way the set can.

</details>

**Q: Why does an LLM hallucinate, in one sentence?**

<details>
<summary>Show answer</summary>

Because training is lossy compression and generation is next-token prediction: it produces what's *plausible*, not what it has looked up and verified.

</details>

---
# Day 5 - Tokens, sampling, and cost

**Big idea:** models read **tokens**, not words, and you pay per token. Count them and you can predict the bill.

## What I built
- `days/day-005/alerts.py` - 20 synthetic alerts (temperature spike, energy spike, device offline), varying tenant, machine ID, severity.
- `days/day-005/token_cost.py` - counts tokens per alert for two models, prints a table, the most/least expensive alert, and the cost per 1,000 alerts.

## The concepts

**A token** is a chunk of text from the model's vocabulary, about **4 English characters** on average. `"Machine M-14 temperature 92C..."` splits like `Machine| M|-|14| temperature| |92|C|...`. Common words = 1 token; rare words = several.

**BPE (byte-pair encoding)** builds the vocabulary: start from bytes (256 symbols), repeatedly merge the most frequent adjacent pair into a new symbol, thousands of times -> ~100k-200k tokens.

**Every model's tokenizer is different**, because each was trained on different text. Same string, different ruler, different count.
- `gpt-4o` -> `o200k_base` (older GPT-4/3.5 -> `cl100k_base`). `tiktoken.encoding_for_model("gpt-4o")` gives the exact tokenizer.
- **tiktoken has no Claude tokenizer.** I counted Claude with `cl100k_base` as a stand-in and **labeled it an estimate**. Exact counts need Anthropic's own `count_tokens` endpoint.

**Context window** = the most tokens a model can see at once (prompt + answer). Over the limit -> the API rejects the request, or something upstream silently truncates (worse). Check length *before* sending.

**Sampling.** The model outputs a probability for every possible next token, then *picks* one:
- **Temperature** reshapes the probabilities. Low (near 0) = almost always the top choice (predictable). High = flatter, surprises win more often.
- **Top-p** keeps only the smallest set of top tokens whose probabilities add up to p (e.g. 0.9), then samples from those. It cuts off the weird tail.
- For alert triage: **low temperature**. The same alert should get the same boring summary every time.

**Why output costs more than input:** the prompt is processed in one parallel pass; output is generated one token at a time, each needing its own pass.

## My numbers

| | gpt-4o (exact, o200k_base) | claude-sonnet-5 (estimate, cl100k_base) |
|---|---|---|
| Total input tokens, 20 alerts | 522 | 522 |
| Average per alert | 26.1 | 26.1 |
| Price per 1M (in / out) | $2.50 / $10.00 | $2.00 / $10.00 |
| **Cost per 1,000 alerts** (with 40 output tokens each) | **$0.4653** | **$0.4522** |

Prices came from each provider's own pricing page (developers.openai.com/api/docs/pricing and claude.com/pricing), checked 2026-09-22 - never from memory.

```
cost per 1,000 = (avg_input x input_price_per_token + avg_output x output_price_per_token) x 1000
gpt-4o:  (26.1 x 2.50/1M + 40 x 10.00/1M) x 1000 = $0.4653
```

**Caveat:** this multiplies an **average** by 1,000. Real billing **sums each request's actual tokens**. Fine here (alerts are 21-29 tokens), misleading when lengths vary a lot.

In [3]:
# Try it: guess the token count before running. Try your own sentences.
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")
for text in ["hello", "Machine M-14 temperature 92C exceeds threshold 85C", "antidisestablishmentarianism"]:
    ids = enc.encode(text)
    print(f"{len(ids):>2} tokens: " + "|".join(enc.decode([i]) for i in ids))

 1 tokens: hello
13 tokens: Machine| M|-|14| temperature| |92|C| exceeds| threshold| |85|C
 6 tokens: ant|idis|est|ablishment|arian|ism


## Questions I was asked

**Q: Why can the same alert string produce a different token count depending on the model's encoding?**

<details>
<summary>Show answer</summary>

Each tokenizer's vocabulary comes from running BPE on that company's own training data, merging whatever pairs were common *there*. So one tokenizer may have "temperature" as a single token while another splits it into pieces. The string is the same; the ruler measuring it is different.

</details>

**Q: Extend token_cost.py to print the most and least expensive alert. What did you get?**

<details>
<summary>Show answer</summary>

Using `max()` / `min()` over the per-alert token counts: most expensive = index 1 (29 tokens, an energy alert); cheapest = index 18 (21 tokens, a temperature alert).

</details>

**Q: What breaks if a user-submitted alert is 50,000 characters and goes straight into the prompt with no length check?**

<details>
<summary>Show answer</summary>

~12,500 tokens from one field. (1) If the total exceeds the context window, the API **rejects** the request - you must handle that failure. (2) If something upstream **silently truncates** instead, you lose the part that mattered and confidently summarize an incomplete alert - worse, because nothing visibly breaks. (3) Even under the limit, one huge input can cost as much as hundreds of normal alerts. So check length first: reject, truncate with a warning, or summarize.

</details>

**Q: Do your two models price input and output the same? Why might a provider price them differently?**

<details>
<summary>Show answer</summary>

Differently: gpt-4o $2.50 in / $10 out (4x); claude-sonnet-5 $2 in / $10 out (5x). Output is generated one token at a time (a full pass per token), while input is processed in one parallel pass, so output costs more compute per token.

</details>

## Karpathy - "Deep Dive into LLMs like ChatGPT", Tokenization -> Inference (10 notes)

1. Text must become numbers. Raw bits make sequences far too long - bad, because sequence length is exactly what the context window limits.
2. Group bits into bytes (256 values), then run BPE: merge the most frequent adjacent pair into a new symbol (IDs from 256 upward).
3. Repeat thousands of times -> a compact vocabulary that still covers any text (~100k symbols for GPT-4).
4. A token isn't a word - it's whatever chunk was frequent enough to earn its own symbol.
5. Vocabularies are learned from specific data, so different models chunk the same string differently.
6. The model never sees characters, only token IDs - which is why spelling and arithmetic can behave oddly.
7. Pretraining: tokenize a huge internet scrape and train the network to predict the next token.
8. Training adjusts weights so predicted next-token probabilities match the data's statistics.
9. The result is a **base model**: great at continuing text, not a helpful assistant.
10. Inference: feed a sequence, sample one token from the distribution, append it, repeat. Because it *samples*, the same prompt can give different continuations.

## Test yourself

**Q: You count a Claude prompt with tiktoken. What must you write next to the number?**

<details>
<summary>Show answer</summary>

That it's an **estimate** (e.g. "approximated with cl100k_base"). tiktoken has no Claude tokenizer; an exact count needs Anthropic's `count_tokens` endpoint.

</details>

**Q: Temperature 0.2 vs 2.0 - which for summarizing alerts, and why?**

<details>
<summary>Show answer</summary>

0.2 (low). You want the same alert to give the same consistent summary every time. High temperature adds randomness - good for creative writing, bad for triage.

</details>

**Q: What does top-p = 0.9 do?**

<details>
<summary>Show answer</summary>

Keeps only the smallest set of most-likely tokens whose probabilities add up to 0.9, and samples from just those - the unlikely tail is thrown away.

</details>

---
# Day 6 - Middleware, logging to Postgres, testing with fakes

**Big idea:** record time, tokens, and cost for every `/chat` call in Postgres - without slowing down or breaking the stream.

## What I built
| File | Job |
|---|---|
| `days/day-006/app.py` | Day 3's `/chat` + `LoggingMiddleware` (raw ASGI) |
| `days/day-006/db.py` | asyncpg connection pool + `insert_request(...)` |
| `days/day-006/cost.py` | `estimate_cost(...)`, importing Day 5's prices |
| `days/day-006/test_middleware.py` | tests with the LLM and the database both faked |
| Postgres table `requests` | `id, ts, provider, model, prompt_tokens, completion_tokens, ttfb_ms, latency_ms, cost_usd` |

No Docker on my Mac, so I used Homebrew Postgres (`brew services start postgresql@18`) instead of `docker run postgres:16`. Same result: a real local Postgres on port 5432.

## The concepts

**Middleware** wraps every request - a guard at the door who sees everything go in and out. Natural place for a stopwatch.

**ASGI in one breath.** The server talks to the app through `scope` (request info: path, query string), `receive` (incoming messages) and `send` (outgoing messages). A response = one `http.response.start` (status + headers) + one or more `http.response.body` messages. Streaming = many body messages over time.

**Raw ASGI middleware: wrap `send`.** Write a class with `async def __call__(self, scope, receive, send)`. Replace `send` with `wrapped_send` that records the time of the first body chunk (**TTFB**), keeps a private copy of the bytes, and **immediately** forwards the message. When `self.app(...)` returns, the stream is over (**total latency**). Nothing is held back.

**Why not `BaseHTTPMiddleware`?** (corrected on Day 7 by measuring)
- The Day 6 brief said it buffers the whole response. On the installed Starlette 1.6.0 **it doesn't** - chunks still arrived at 0.3 / 0.6 / 0.9 s.
- The real problem: `call_next` returns as soon as the response **starts** (headers). The body streams **after** `dispatch` returns. A timer around `call_next` read **0.00 s** for a 0.92 s stream.
- To time or read the body there you'd have to consume `body_iterator` yourself - and reading it all before returning **does** buffer the stream. Raw ASGI avoids all of this.

**Logging must never break the request.** The insert runs **after** the response is fully sent, inside `try/except Exception`. Postgres down = one missing log row, never a failed answer. Observability must not take down the thing it observes.

**`db.py`:**
- **Connection pool** - opening connections is slow, so keep a few and reuse them. Created lazily on the first insert.
- **Placeholders `$1 ... $7`** - values travel separately from the SQL text, so a prompt like `'); DROP TABLE requests; --` is stored as harmless text. Never build SQL with f-strings.
- `DATABASE_URL` comes from `.env`.

**`cost.py` reuses Day 5's prices** by importing them - one source of truth. A model Day 5 never priced (`llama3.2:3b`, Gemini) returns `0.0`, meaning **untracked**, not free.

**Counting tokens after the stream.** The middleware sees raw SSE bytes, so it extracts the `data:` lines with a regex and counts them with tiktoken.
**Bug found on Day 7:** sse-starlette ends lines with `\r\n`; the regex kept the `\r`, which **doubled** completion tokens and cost. Fixed with `rb"^data:[ ]?(.*?)\r?$"`. The old test only checked `completion_tokens > 0`, so it passed anyway - now it asserts the exact number.

**Testing with fakes (no network, no DB):**
- Fake LLM: `monkeypatch.setitem(TOKEN_SOURCES, "ollama", fake_tokens)` where `fake_tokens` yields `"pong", " ", "pong"`.
- Fake DB: `monkeypatch.setattr(app_module, "insert_request", AsyncMock())`, then check what it was called with.
- **Trap:** Day 3 and Day 6 both have `app.py`, and Python caches modules by name. Running all tests together, Day 6's `import app` got Day 3's app. Fix: load the file under a private name with `importlib.util.spec_from_file_location`.

**p95 latency** = 95% of requests were faster than this. The median shows the typical request; p95 shows the slow tail users complain about.

### Key code (the middleware, simplified)
```python
class LoggingMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http" or scope["path"] != "/chat":
            return await self.app(scope, receive, send)

        start, ttfb, status, chunks = time.perf_counter(), None, None, []

        async def wrapped_send(message):
            nonlocal ttfb, status
            if message["type"] == "http.response.start":
                status = message["status"]
            elif message["type"] == "http.response.body":
                if ttfb is None:
                    ttfb = time.perf_counter() - start
                chunks.append(message.get("body", b""))
            await send(message)                           # forward immediately

        await self.app(scope, receive, wrapped_send)
        latency = time.perf_counter() - start
        if status != 200:
            return                                        # don't log errors like 400
        # ...extract text, count tokens, estimate cost...
        try:
            await insert_request(...)
        except Exception as exc:
            print(f"failed to log request: {exc}")        # never break the request
```

```sql
SELECT provider,
       percentile_cont(0.95) WITHIN GROUP (ORDER BY latency_ms) AS p95_latency_ms
FROM requests
GROUP BY provider;
```

## Questions I was asked

**Q: Why does BaseHTTPMiddleware fight you on a streaming SSE endpoint when it works fine on JSON?**

<details>
<summary>Show answer</summary>

`call_next` returns when the response **starts**; for JSON the body is one quick chunk so nothing is missed, but for SSE the entire answer streams *after* that point. Timing or reading the body inside `dispatch` gets it wrong (the timer read 0.00 s), and reading the full body before returning it buffers the stream, which destroys TTFT.
(My original Day 6 answer said it always buffers the whole response - measurement on Day 7 showed that's not what the installed version does.)

</details>

**Q: Write the SQL for p95 latency_ms grouped by provider.**

<details>
<summary>Show answer</summary>

```sql
SELECT provider,
       percentile_cont(0.95) WITHIN GROUP (ORDER BY latency_ms) AS p95_latency_ms
FROM requests
GROUP BY provider;
```

</details>

**Q: Postgres is down when a /chat request comes in. Does the SSE response still stream? Which behavior do you want?**

<details>
<summary>Show answer</summary>

It still streams fully. The insert only happens after `self.app(...)` returns (every chunk already sent), and it's wrapped in `try/except`, including the lazy pool creation. That's the behavior you want: losing a log row is far cheaper than failing the user's answer. The middleware must (a) log after the response, not during it, and (b) catch broadly around the insert.

</details>

**Q: What goes wrong if prompt_tokens only counts the user's raw text, once you add a system prompt and few-shot examples?**

<details>
<summary>Show answer</summary>

The middleware only sees the raw `prompt` from the URL, not the full string the app builds and sends to the model. Add a system prompt and few-shot examples and the real input grows (11.6x in my demo) while the logged number doesn't move - so cost and any budget alerts quietly under-report. Fix: count tokens on the exact string sent to the model.

</details>

### Recall quiz from Day 6

**Q: (Day 5) Why can gpt-4o and claude-sonnet-5 report different token counts for the same string?**

<details>
<summary>Show answer</summary>

Different vocabularies, built by BPE on different training data.

</details>

**Q: (Day 5) Which is priced higher per token, input or output, and why?**

<details>
<summary>Show answer</summary>

Output: generated one token at a time, versus one parallel pass for the whole input.

</details>

**Q: (Day 3) What specifically triggers the asyncio.CancelledError in your SSE generator?**

<details>
<summary>Show answer</summary>

sse-starlette's disconnect listener sees the client leave and cancels the task running the generator.

</details>

**Q: (Day 3) Why doesn't request.is_disconnected() fire in your generator?**

<details>
<summary>Show answer</summary>

Each ASGI message can be read only once, and sse-starlette's own listener reads the `http.disconnect` message first, so my check never sees it.

</details>

## Test yourself

**Q: In raw ASGI middleware, where exactly do you record TTFB and total latency?**

<details>
<summary>Show answer</summary>

TTFB: inside `wrapped_send`, the first time a `http.response.body` message passes through. Total latency: right after `await self.app(scope, receive, wrapped_send)` returns.

</details>

**Q: Why use $1 placeholders instead of an f-string in the INSERT?**

<details>
<summary>Show answer</summary>

Values are sent separately from the SQL, so user text can never be executed as SQL (prevents SQL injection) and quoting is handled correctly.

</details>

**Q: Why is `assert completion_tokens > 0` a weak test?**

<details>
<summary>Show answer</summary>

It passes for any positive number, including wrong ones - it passed while the count was doubled by the `\r` bug. Assert the exact expected value.

</details>

---
# Day 7 - Anagrams: counting and canonical keys

**Big idea:** anagrams = same letters, same amounts. Either **count** the letters or **sort** them, and "is this a rearrangement?" becomes a simple equality check.

## Valid Anagram (LeetCode 242) - `days/day-007/valid_anagram.py`

1. **One-line check first:** different lengths -> not anagrams.
2. **Counting:** count each character of `s` in a dict; walk `t` and subtract. If a count is already 0 when `t` needs it -> False.
3. **Sorting:** `sorted(s) == sorted(t)`.

| Approach | Time | Space |
|---|---|---|
| Counting (dict) | O(n) | O(k), which is O(1) for a-z (k <= 26) |
| Sorting | O(n log n) | O(n) (sorted() builds new lists) |

n = string length, k = number of distinct characters.

**Likely mistake:** only checking that each letter *exists* instead of subtracting counts. That wrongly accepts `"aacc"` vs `"ccac"` (same letters, different amounts).

**Unicode follow-up:**
- A 26-slot array breaks; a **dict** handles any character (space becomes O(k)).
- `é` can be one code point or `e` + a combining accent - they look identical but compare unequal. Normalize with `unicodedata.normalize("NFC", s)`.
- Case-insensitive? Use `casefold()`. It can change length (`"Straße"` -> `"strasse"`, 6 -> 7), so do the **length check after normalizing**.
- Emoji made of several code points (joined by zero-width joiners) need grapheme-cluster counting (e.g. the `regex` library's `\X`).

## Group Anagrams (LeetCode 49) - `days/day-007/group_anagrams.py`

Give every word a **canonical key** that all its anagrams share, and group in a dict `key -> [words]`.

| Key | Example for "eat" | Time | Space |
|---|---|---|---|
| Sorted letters | `"aet"` | O(n · k log k) | O(n · k) |
| 26-count tuple (I used this) | `(1,0,0,0,1,...,1,...)` | O(n · k) | O(n · k) |

n = number of strings, k = length of the longest string. The count key is **asymptotically faster**, but when I timed it, sorting won for short words (C-speed `sorted()` vs a Python loop) and counting won for long ones (0.19 s vs 0.51 s at k = 1000). **Big-O shows growth, not who wins at every size.**

**The trap: a list can't be a dict key.** Dict keys must be **hashable**: the dict files each key by its hash and trusts it never changes. A list is mutable, so Python raises `TypeError: unhashable type: 'list'`. Use `tuple(counts)`.

**Sneakier mistake:** joining counts into a string with no separator. `[1, 11]` and `[11, 1]` both become `"111"` - two different words silently grouped together. Use a tuple, or a separator like `"#"`.

In [4]:
# Try it - these are my real Day 7 solutions.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "days" / "day-007"))
from valid_anagram import is_anagram, is_anagram_sorted
from group_anagrams import group_anagrams

print(is_anagram("anagram", "nagaram"), is_anagram("aacc", "ccac"), is_anagram_sorted("", ""))
print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))

True False True
[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


## Recall quiz from Day 7

**Q: (Day 6) Why does a timer around call_next in BaseHTTPMiddleware give the wrong latency for SSE?**

<details>
<summary>Show answer</summary>

`call_next` returns when the response starts (headers ready); the body streams afterwards. Measured: 0.00 s on the timer for a 0.92 s stream. Time the stream where chunks are actually sent.

</details>

**Q: (Day 6) Why must a logging failure never fail /chat, and how did you enforce it?**

<details>
<summary>Show answer</summary>

Monitoring must never take down the feature it monitors. Insert runs after the response is fully sent, inside `try/except Exception`.

</details>

**Q: (Day 4) Two Sum: what are the hash map's keys and values, and why does one pass work?**

<details>
<summary>Show answer</summary>

Keys = numbers seen so far, values = their indices. When you reach the second number of a pair, the first is already stored. Check before storing so a number can't pair with itself.

</details>

**Q: (Day 4) Contains Duplicate: hash set vs sorting - what's the trade?**

<details>
<summary>Show answer</summary>

Memory for speed: set = O(n) time, O(n) memory; sort = O(n log n) time, little extra memory, but mutates the input and can't stop early.

</details>

## Test yourself

**Q: Why do the length check AFTER Unicode normalization and casefolding?**

<details>
<summary>Show answer</summary>

Because normalization and casefolding can change the length ("Straße" is 6, "strasse" is 7). Checking first could reject strings that are anagrams after normalizing.

</details>

**Q: Write the time complexity of both Group Anagrams keys. Which is faster asymptotically?**

<details>
<summary>Show answer</summary>

Sorted key: O(n · k log k). Count key: O(n · k). The count key is faster asymptotically (no log k), though sorting can win on short words in practice.

</details>

---
# Patterns that keep coming back

1. **Measure, don't assume.** Median of 3 (Day 2), TTFT (Day 3), p95 (Day 6), timing both anagram keys (Day 7), and testing the `BaseHTTPMiddleware` claim - which turned out to be wrong.
2. **Label what you don't know.** Claude token counts are *estimates*; unpriced models are *untracked* (0.0), not free.
3. **Secondary systems must never break the primary one.** Logging failures are caught; the user's answer always wins.
4. **Never wait forever.** Every network call gets a timeout (Day 3).
5. **Stop work nobody wants.** On disconnect, cancel the upstream call (Day 3).
6. **"Have I seen this before?" -> hash set / hash map.** Days 4 and 7.
7. **One source of truth.** Import prices, don't retype them (Day 6).
8. **Strong tests assert exact values** and don't need the network or a database (Day 6).
9. **Secrets stay out of git**, and a leaked key gets rotated first (Day 1).
10. **Output tokens cost more than input tokens**, and cost = tokens x price (Days 2 and 5).

---
# Final self-test

Try all of these without scrolling up. Mark the ones you missed and re-read that day.

**Q: What's the difference between pyproject.toml and uv.lock?**

<details>
<summary>Show answer</summary>

pyproject = what you want (ranges); uv.lock = exact versions of everything, including transitive dependencies. Commit both.

</details>

**Q: First thing to do after pushing a secret?**

<details>
<summary>Show answer</summary>

Rotate/revoke the key. Cleaning git history comes after.

</details>

**Q: Formula for the cost of one LLM call?**

<details>
<summary>Show answer</summary>

input_tokens/1M x input_price + output_tokens/1M x output_price.

</details>

**Q: Why the median rather than the average of 3 runs?**

<details>
<summary>Show answer</summary>

One outlier drags the average; the median ignores it.

</details>

**Q: What does TTFT stand for, and why does it matter?**

<details>
<summary>Show answer</summary>

Time to first token. It's how long the user stares at a blank screen - what they actually feel.

</details>

**Q: What content type does an SSE response use?**

<details>
<summary>Show answer</summary>

`text/event-stream`.

</details>

**Q: What happens in your generator when the user closes the tab?**

<details>
<summary>Show answer</summary>

sse-starlette cancels the task -> `asyncio.CancelledError` is raised; catch it, clean up, re-raise.

</details>

**Q: Why set an httpx timeout?**

<details>
<summary>Show answer</summary>

Without it a silent upstream hangs the task forever and leaks connections/memory.

</details>

**Q: Set or dict: Contains Duplicate? Two Sum?**

<details>
<summary>Show answer</summary>

Contains Duplicate: set (only 'seen?'). Two Sum: dict (need the index too).

</details>

**Q: Roughly how many English characters per token?**

<details>
<summary>Show answer</summary>

About 4.

</details>

**Q: Which tiktoken encoding does gpt-4o use?**

<details>
<summary>Show answer</summary>

`o200k_base`.

</details>

**Q: Low or high temperature for alert triage?**

<details>
<summary>Show answer</summary>

Low - consistent, predictable output.

</details>

**Q: Where does raw ASGI middleware record TTFB?**

<details>
<summary>Show answer</summary>

In the wrapped `send`, at the first `http.response.body` message.

</details>

**Q: Why log AFTER the response instead of before?**

<details>
<summary>Show answer</summary>

So the user already has the full answer; a logging failure can't affect it.

</details>

**Q: What does p95 latency tell you that the median doesn't?**

<details>
<summary>Show answer</summary>

How slow the slow tail is - 95% of requests were faster than it.

</details>

**Q: Why can't a list be a dict key?**

<details>
<summary>Show answer</summary>

It's mutable, so its hash could change; Python rejects unhashable keys. Use a tuple.

</details>

**Q: Group Anagrams: two canonical keys and their time complexity?**

<details>
<summary>Show answer</summary>

Sorted letters O(n·k log k); 26-count tuple O(n·k).

</details>

**Q: Valid Anagram: the one-line check before counting?**

<details>
<summary>Show answer</summary>

`if len(s) != len(t): return False`.

</details>

**Q: Why did the Day 6 middleware count twice the real completion tokens?**

<details>
<summary>Show answer</summary>

sse-starlette uses `\r\n`; the regex kept each `\r`, and each became extra tokens.

</details>

**Q: What's an LLM, physically?**

<details>
<summary>Show answer</summary>

Two files: the weights and a small program that runs them.

</details>

---
**You made it through Days 1-7.** From here on: attempt first, then check. Come back to this file whenever something from an earlier day feels fuzzy.